Cell 1 — Imports + quick sanity checks

In [1]:
import os  # file/path checks
import time  # timing utilities

import cv2  # video loading + drawing
import mediapipe as mp  # mediapipe main package

print("✅ Imported cv2 and mediapipe")  # track
print("OpenCV version:", cv2.__version__)  # track
print("mediapipe path:", mp.__file__)  # track
print("Has mp.solutions?:", hasattr(mp, "solutions"))  # track
print("Has mp.tasks?:", hasattr(mp, "tasks"))  # track

VIDEO_PATH = "video.MOV"  # your dataset video filename

print("📁 CWD:", os.getcwd())  # track
print("🎬 Video path:", os.path.abspath(VIDEO_PATH))  # track
print("✅ Video exists?:", os.path.exists(VIDEO_PATH))  # track

print("Shadow check - mediapipe.py exists here?:", os.path.exists("mediapipe.py"))  # avoid name shadowing
print("Shadow check - mediapipe folder exists here?:", os.path.isdir("mediapipe"))  # avoid name shadowing


✅ Imported cv2 and mediapipe
OpenCV version: 4.12.0
mediapipe path: C:\Users\HP\anaconda3\envs\DL\Lib\site-packages\mediapipe\__init__.py
Has mp.solutions?: False
Has mp.tasks?: True
📁 CWD: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing
🎬 Video path: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing\video.MOV
✅ Video exists?: True
Shadow check - mediapipe.py exists here?: False
Shadow check - mediapipe folder exists here?: False


Cell 2 — Download the face detector model

In [2]:
import urllib.request  # download helper

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_detector/"
    "blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
)  # model download URL

MODEL_PATH = "blaze_face_short_range.tflite"  # local model file name

print("📦 Model path:", os.path.abspath(MODEL_PATH))  # track

if not os.path.exists(MODEL_PATH):  # check if model already exists
    print("⬇️ Downloading face detector model...")  # track
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)  # download model
    print("✅ Model downloaded")  # track
else:
    print("✅ Model already exists (skip download)")  # track


📦 Model path: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing\blaze_face_short_range.tflite
✅ Model already exists (skip download)


Cell 3 — Create MediaPipe Tasks FaceDetector options

In [3]:
from mediapipe.tasks.python import vision  # MediaPipe vision tasks

print("✅ Imported mediapipe.tasks.python.vision")  # track

BaseOptions = mp.tasks.BaseOptions  # base options class
FaceDetectorOptions = vision.FaceDetectorOptions  # face detector options class
RunningMode = vision.RunningMode  # running mode enum

print("⚙️ Creating FaceDetector options...")  # track

options = FaceDetectorOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),  # set model file path
    running_mode=RunningMode.VIDEO,  # VIDEO mode requires timestamps
    min_detection_confidence=0.5,  # ignore weak detections
    min_suppression_threshold=0.3  # suppress overlapping boxes
)

print("✅ FaceDetector options created")  # track


✅ Imported mediapipe.tasks.python.vision
⚙️ Creating FaceDetector options...
✅ FaceDetector options created


Cell 4 — Function: Load video, detect faces, draw bounding boxes

In [4]:
def run_video_face_detection(
    video_path="video.MOV",  # input video path
    window_name="Face Detection",  # display window name
    print_every=30,  # print status every N frames
    save_output=False,  # optionally save output video
    output_path="video_annotated.mp4"  # output filename
):
    print("🎬 Starting video face detection...")  # track
    print("📽️ Input video:", os.path.abspath(video_path))  # track

    if not os.path.exists(video_path):  # check video exists
        print("❌ ERROR: Video file not found:", video_path)  # track
        return  # stop

    cap = cv2.VideoCapture(video_path)  # open the video file

    if not cap.isOpened():  # verify it opened
        print("❌ ERROR: Could not open video:", video_path)  # track
        return  # stop

    fps = cap.get(cv2.CAP_PROP_FPS)  # read FPS from metadata
    if fps is None or fps <= 0:  # handle missing/invalid FPS
        fps = 30.0  # fallback FPS
        print("⚠️ WARNING: FPS not found, using fallback FPS=30")  # track
    else:
        print("✅ Video FPS:", fps)  # track

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  # frame width
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # frame height
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # total frames (may be 0 for some codecs)

    print("🧾 Video size:", width, "x", height)  # track
    print("🧮 Total frames (may be 0 if unknown):", total_frames)  # track

    writer = None  # video writer placeholder
    if save_output:  # if user wants to save annotated output
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # mp4 codec
        writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))  # create writer
        print("💾 Saving annotated output to:", os.path.abspath(output_path))  # track

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)  # create window
    cv2.moveWindow(window_name, 50, 50)  # move window

    frame_count = 0  # processed frame counter
    t0 = time.perf_counter()  # timing start

    try:
        with vision.FaceDetector.create_from_options(options) as detector:  # create detector
            print("✅ FaceDetector initialized")  # track

            while True:
                ok, frame_bgr = cap.read()  # read next frame
                if not ok:  # end or read error
                    print("🏁 End of video or failed to read frame. Stopping...")  # track
                    break  # exit loop

                frame_count += 1  # increment frame count

                # timestamp in ms for VIDEO mode (based on frame index and fps)
                timestamp_ms = int((frame_count / fps) * 1000)  # compute timestamp

                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)  # convert to RGB
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)  # wrap for mediapipe

                result = detector.detect_for_video(mp_image, timestamp_ms)  # run detection

                faces_found = 0  # track faces per frame
                if result.detections:  # if faces detected
                    faces_found = len(result.detections)  # count faces

                    for det in result.detections:  # loop each face detection
                        bbox = det.bounding_box  # get bounding box

                        x = int(bbox.origin_x)  # bbox left
                        y = int(bbox.origin_y)  # bbox top
                        w = int(bbox.width)  # bbox width
                        h = int(bbox.height)  # bbox height

                        cv2.rectangle(frame_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)  # draw box

                        score = det.categories[0].score if det.categories else 0.0  # confidence
                        cv2.putText(
                            frame_bgr,  # draw on BGR frame
                            f"{score:.2f}",  # show score
                            (x, max(0, y - 10)),  # label position
                            cv2.FONT_HERSHEY_SIMPLEX,  # font
                            0.6,  # font scale
                            (0, 255, 0),  # color
                            2  # thickness
                        )

                # print progress every N frames
                if frame_count % print_every == 0:
                    elapsed = time.perf_counter() - t0  # elapsed seconds
                    print(f"📌 frame={frame_count} | ts={timestamp_ms}ms | faces={faces_found} | elapsed={elapsed:.2f}s")  # track

                cv2.imshow(window_name, frame_bgr)  # display frame

                if writer is not None:  # if saving output
                    writer.write(frame_bgr)  # write annotated frame

                # allow quit by closing window
                if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                    print("🛑 Window closed (X). Exiting...")  # track
                    break  # stop

                key = cv2.waitKey(1) & 0xFF  # poll keyboard
                if key == ord('q') or key == 27:  # q or ESC
                    print("🛑 Quit key pressed (q or ESC). Exiting...")  # track
                    break  # stop

    except KeyboardInterrupt:
        print("🛑 Interrupted from Jupyter (Kernel → Interrupt). Exiting...")  # track

    finally:
        cap.release()  # release video handle
        if writer is not None:  # if writer created
            writer.release()  # release writer
        cv2.destroyAllWindows()  # close windows
        print("✅ Video released and windows closed")  # track


Cell 5 — Run it on your dataset video

In [5]:
print("🚀 Running detection on dataset video...")  # track

run_video_face_detection(
    video_path=VIDEO_PATH,  # use your dataset video
    window_name="Dataset Face Detection",  # window title
    print_every=30,  # progress prints
    save_output=False,  # set True to save annotated video
    output_path="video_annotated.mp4"  # output file name if saving
)

print("✅ Done")  # track


🚀 Running detection on dataset video...
🎬 Starting video face detection...
📽️ Input video: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing\video.MOV
✅ Video FPS: 29.97002997002997
🧾 Video size: 1920 x 1080
🧮 Total frames (may be 0 if unknown): 12867
✅ FaceDetector initialized
📌 frame=30 | ts=1001ms | faces=1 | elapsed=0.90s
📌 frame=60 | ts=2002ms | faces=1 | elapsed=1.49s
📌 frame=90 | ts=3003ms | faces=1 | elapsed=1.97s
📌 frame=120 | ts=4004ms | faces=1 | elapsed=2.45s
📌 frame=150 | ts=5005ms | faces=1 | elapsed=2.93s
📌 frame=180 | ts=6006ms | faces=1 | elapsed=3.41s
📌 frame=210 | ts=7007ms | faces=1 | elapsed=3.88s
📌 frame=240 | ts=8008ms | faces=1 | elapsed=4.35s
📌 frame=270 | ts=9009ms | faces=1 | elapsed=4.83s
📌 frame=300 | ts=10010ms | faces=1 | elapsed=5.31s
📌 frame=330 | ts=11011ms | faces=1 | elapsed=5.78s
📌 frame=360 | ts=12012ms | faces=1 | elapsed=6.26s
📌 frame=390 | ts=13013ms | faces=1 | elapsed=6.75s
📌 frame=420 | ts=14014ms | faces=1 |

In [ ]:
import matplotlib.pyplot as plt  # display frames in notebook

print("🧪 Loading video to show first 5 annotated frames...")  # track

cap = cv2.VideoCapture(VIDEO_PATH)  # open the dataset video

if not cap.isOpened():  # check if video opened
    print("❌ ERROR: Could not open video:", VIDEO_PATH)  # track
else:
    fps = cap.get(cv2.CAP_PROP_FPS)  # get FPS
    if fps is None or fps <= 0:  # handle invalid FPS
        fps = 30.0  # fallback FPS
        print("⚠️ FPS not found, using fallback FPS=30")  # track
    else:
        print("✅ Video FPS:", fps)  # track

    annotated_frames = []  # store annotated frames for display
    face_counts = []  # store number of faces per frame

    with vision.FaceDetector.create_from_options(options) as detector:  # create detector
        print("✅ FaceDetector initialized for preview")  # track

        frame_count = 0  # frame counter

        while frame_count < 5:  # only process first 5 frames
            ok, frame_bgr = cap.read()  # read one frame
            if not ok:  # stop if video ended early
                print("🏁 Video ended before 5 frames were read")  # track
                break  # exit loop

            frame_count += 1  # increment frame counter

            timestamp_ms = int((frame_count / fps) * 1000)  # compute timestamp for VIDEO mode

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)  # convert BGR -> RGB
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)  # wrap for mediapipe

            result = detector.detect_for_video(mp_image, timestamp_ms)  # run detection

            faces_found = 0  # default face count
            if result.detections:  # if detections exist
                faces_found = len(result.detections)  # count faces

                for det in result.detections:  # loop through detections
                    bbox = det.bounding_box  # get bounding box

                    x = int(bbox.origin_x)  # left
                    y = int(bbox.origin_y)  # top
                    w = int(bbox.width)  # width
                    h = int(bbox.height)  # height

                    cv2.rectangle(frame_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)  # draw bbox

                    score = det.categories[0].score if det.categories else 0.0  # confidence
                    cv2.putText(  # write scorez
                        frame_bgr,  # draw on frame
                        f"{score:.2f}",  # score text
                        (x, max(0, y - 10)),  # text position
                        cv2.FONT_HERSHEY_SIMPLEX,  # font
                        0.6,  # font scale
                        (0, 255, 0),  # color
                        2  # thickness
                    )

            face_counts.append(faces_found)  # store face count

            annotated_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)  # convert for matplotlib display
            annotated_frames.append(annotated_rgb)  # store annotated frame

            print(f"📌 Frame {frame_count}/5 | ts={timestamp_ms}ms | faces={faces_found}")  # track

    cap.release()  # release the video
    print("✅ Released video after preview")  # track

    # display the annotated frames
    for i, img in enumerate(annotated_frames):  # loop stored frames
        plt.figure()  # new figure per frame
        plt.imshow(img)  # show frame
        plt.title(f"Annotated Frame {i+1} | faces={face_counts[i]}")  # title with face count
        plt.axis("off")  # hide axes

    plt.show()  # render all figures
    print("✅ Displayed first annotated frames")  # track


🧪 Loading video to show first 5 annotated frames...
✅ Video FPS: 29.97002997002997
✅ FaceDetector initialized for preview
📌 Frame 1/5 | ts=33ms | faces=1
📌 Frame 2/5 | ts=66ms | faces=1
📌 Frame 3/5 | ts=100ms | faces=1
📌 Frame 4/5 | ts=133ms | faces=1
📌 Frame 5/5 | ts=166ms | faces=1
✅ Released video after preview
